# 作业 02：把多个输入接入上周的训练模板

作业 01 比较了同一个样本的两种表示：两个标量输入张量，或一个包含两个数的输入张量。本题把多个样本分别放入 x1、x2，练习补全训练代码。

请参考上周模板，让同一个预测函数同时使用 x1 和 x2。所有样本共享 w1、w2、b；每次预测都需要两个输入。下面提供函数和数据，参数创建、损失、梯度与更新由你完成。

本文件可以独立运行，不依赖作业 01 的内核变量。


In [1]:
import torch

def predict(x1, x2, w1, w2, b):
    return w1 * x1 + w2 * x2 + b


## 训练数据：每一行是同一个样本的两个输入和一个目标

下面 x1、x2、y 的 shape 都为 `(6,)`，表示 6 个样本。`predict(x1, x2, ...)` 同时使用两个输入张量，返回 6 个预测值。

| 样本 i | x1[i] | x2[i] | y[i] |
|---|---|---|---|
| 0 | -2 | 1 | -6 |
| 1 | -1 | 0 | -1 |
| 2 | 0 | 2 | -5 |
| 3 | 1 | -1 | 6 |
| 4 | 2 | 3 | -4 |
| 5 | 3 | 1 | 4 |

训练时不要修改输入和目标，也不要直接填写猜测的参数答案。


In [2]:
x1 = torch.tensor([-2., -1., 0., 1., 2., 3.])
x2 = torch.tensor([1., 0., 2., -1., 3., 1.])
y = torch.tensor([-6., -1., -5., 6., -4., 4.])
print('x1:', x1, 'shape:', tuple(x1.shape))
print('x2:', x2, 'shape:', tuple(x2.shape))
print('y: ', y, 'shape:', tuple(y.shape))


x1: tensor([-2., -1.,  0.,  1.,  2.,  3.]) shape: (6,)
x2: tensor([ 1.,  0.,  2., -1.,  3.,  1.]) shape: (6,)
y:  tensor([-6., -1., -5.,  6., -4.,  4.]) shape: (6,)


## 你的训练区：修改上周模板

创建三个浮点标量参数 w1、w2、b，都启用梯度，并一起交给优化器。每轮调用给定的 predict 函数，同时传入 x1、x2，得到 shape 为 `(6,)` 的 y_pred。

使用绝对值损失：先得到 `(6,)` 的逐样本误差，再用 `.mean()` 得到 shape 为 `()` 的平均损失。只对这个平均损失 backward 一次，然后更新全部参数。

可以从三个参数都为 0、2000 轮、学习率 `0.1 / (epoch ** 0.5)` 开始。这里预测公式是线性的，零初始化与上周的平方参数例子不同。

补完后删除 NotImplementedError。


In [3]:
# 1. 创建浮点标量参数，零初始化，并启用自动求导
w1 = torch.tensor(0.0, requires_grad=True)
w2 = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
optimizer = torch.optim.SGD([w1, w2, b], lr=0.1)

with torch.no_grad():
    initial_loss = torch.abs(y - predict(x1, x2, w1, w2, b)).mean().item()
print(f'训练前的平均损失：{initial_loss:.6f}')

# 2. 每轮执行：预测、损失、清空梯度、反向传播、更新参数
for epoch in range(1, 2001):
    learning_rate = 0.1 / (epoch ** 0.5)
    for param_group in optimizer.param_groups:
        param_group['lr'] = learning_rate
    optimizer.zero_grad()

    y_pred = predict(x1, x2, w1, w2, b)
    sample_losses = torch.abs(y - y_pred)  # (6,)：每个样本一个误差
    loss = sample_losses.mean()           # ()：对六个样本取平均
    loss.backward()                       # 每轮只反向传播一次
    if epoch == 1:
        print('首轮梯度（w1、w2、b）：',
              [parameter.grad.item() for parameter in [w1, w2, b]])
    optimizer.step()                      # 同时更新三个参数

    if epoch == 1 or epoch % 500 == 0:
        print(f'epoch={epoch:4d}, lr={learning_rate:.6f}, '
              f'本轮更新前的平均损失={loss.item():.6f}')

# 3. 使用最后一次更新后的参数，重新计算预测和损失
with torch.no_grad():
    y_pred = predict(x1, x2, w1, w2, b)
    sample_losses = torch.abs(y - y_pred)
    loss = sample_losses.mean()

    print('\n训练后的结果：')
    for name, value in [('w1', w1), ('w2', w2), ('b', b),
                        ('x1', x1), ('x2', x2), ('y', y),
                        ('y_pred', y_pred), ('逐样本损失', sample_losses),
                        ('平均损失', loss)]:
        print(name, 'shape =', tuple(value.shape), 'value =', value)
    print(f'最终参数：w1={w1.item():.6f}, w2={w2.item():.6f}, b={b.item():.6f}')
    print(f'最终平均绝对误差：{loss.item():.8f}')


训练前的平均损失：4.333333
首轮梯度（w1、w2、b）： [-0.8333333730697632, 1.0, 0.3333333134651184]
epoch=   1, lr=0.100000, 本轮更新前的平均损失=4.333333
epoch= 500, lr=0.004472, 本轮更新前的平均损失=0.245472
epoch=1000, lr=0.003162, 本轮更新前的平均损失=0.003635


epoch=1500, lr=0.002582, 本轮更新前的平均损失=0.002817
epoch=2000, lr=0.002236, 本轮更新前的平均损失=0.003900

训练后的结果：
w1 shape = () value = tensor(1.9990, requires_grad=True)
w2 shape = () value = tensor(-3.0010, requires_grad=True)
b shape = () value = tensor(0.9994, requires_grad=True)
x1 shape = (6,) value = tensor([-2., -1.,  0.,  1.,  2.,  3.])
x2 shape = (6,) value = tensor([ 1.,  0.,  2., -1.,  3.,  1.])
y shape = (6,) value = tensor([-6., -1., -5.,  6., -4.,  4.])
y_pred shape = (6,) value = tensor([-5.9997, -0.9996, -5.0026,  5.9994, -4.0056,  3.9954])
逐样本损失 shape = (6,) value = tensor([0.0003, 0.0004, 0.0026, 0.0006, 0.0056, 0.0046])
平均损失 shape = () value = tensor(0.0024)
最终参数：w1=1.999007, w2=-3.001009, b=0.999367
最终平均绝对误差：0.00237277


## 我的观察与实验

1. 现在一次预测需要几个输入张量？x1 和 x2 与样本数 6 分别是什么关系？（10分）
2. 对照上周：预测公式、参数列表和损失计算分别改了哪里？优化器的基本步骤变了吗？（10分）
3. 记录最终参数、预测和平均损失，说明如何判断训练有效。（20分）
4. 为什么这次 w1=w2=b=0 仍然可以开始学习？（10分）

1. 一次预测需要 `x1`、`x2` 两个输入张量。它们的 shape 都是 `(6,)`，分别保存 6 个样本的第一个和第二个特征；`x1[i]`、`x2[i]` 与 `y[i]` 属于同一个样本。这里的两个张量对应两种特征，每个张量中的 6 个数才对应 6 个样本。一次调用返回 6 个预测值。

2. 预测公式改为 $\hat y=w_1x_1+w_2x_2+b$，同时使用两个输入，每个特征有自己的权重。参数列表为 `[w1, w2, b]`，三个参数均为可求梯度的浮点标量，并一起交给 SGD 优化器。损失先计算 `torch.abs(y-y_pred)`，得到 shape 为 `(6,)` 的逐样本绝对误差，再调用 `.mean()` 得到 shape 为 `()` 的平均损失。优化器的基本步骤保持为清空旧梯度、对平均损失反向传播一次、更新全部参数；本题每轮将学习率设为 `0.1 / sqrt(epoch)`。

3. 从三个参数均为 `0` 开始，训练 2000 轮后，得到 `w1≈1.999007`、`w2≈-3.001009`、`b≈0.999367`。最终预测为 `[-5.9997, -0.9996, -5.0026, 5.9994, -4.0056, 3.9954]`，对应真实目标 `[-6, -1, -5, 6, -4, 4]`；逐样本绝对误差约为 `[0.0003, 0.0004, 0.0026, 0.0006, 0.0056, 0.0046]`，平均绝对误差约为 **`0.00237277`**。平均损失从训练前的 `4.333333` 明显降低，而且六个样本的预测均接近目标，说明训练有效。学到的关系接近 $y=2x_1-3x_2+1$。绝对值损失在误差为零处不可导，PyTorch 在该点取零次梯度；附近使用有限学习率更新时仍可能有小幅波动，因此最后损失不必恰好为零。

4. 本题参数在线性项中出现：$\partial\hat y/\partial w_1=x_1$、$\partial\hat y/\partial w_2=x_2$、$\partial\hat y/\partial b=1$，这些导数不因参数初始化为 0 而全部消失。对本题数据，零初始化时三个参数的损失梯度分别约为 `-0.8333、1.0000、0.3333`，因此第一次更新就能改变参数。上周的平方参数例子若含有 $w^2$，对应导数 $2w$ 在 `w=0` 时为 0，可能无法从该点开始更新；本题没有这个问题。
